# What a Probabilistic Forecast Is Actually Worth

## Objectives

Chapter 2 found a real, checked way to share information across Chapter 4's
customer archetypes without training one model per customer. Every result so
far has been a point forecast, though, a single number with no attached
sense of how wrong it might be. A real deployment decision often depends on
that missing piece: a risk-of-exceedance question needs a full density, a
transformer-sizing question needs a specific quantile, and a compliance
statement needs a guarantee that holds regardless of whether the model
itself is well-specified. This chapter builds one real backbone, MLPGAM,
into three different probabilistic paradigms, checks what each is actually
worth, and tests directly whether Chapter 2's own archetypes help here too,
or whether that idea, real and checked in Chapter 2, simply does not
transfer to this different question.

By the end you will be able to:

- Train a real parametric distributional forecast (`MLPGAMNormal`) and a
  real quantile forecast (`MLPGAMFpqr`) on the same 342-customer AusNet pool.
- Wrap either backbone in a genuine split-conformal calibration, the same
  finite-sample coverage guarantee Chapter 2 already used, and check whether
  an uncalibrated model's own stated uncertainty can be trusted as-is.
- Check honestly whether calibrating per Chapter 2's own archetypes sharpens
  a conformal interval, or whether it does not, and explain the real
  mechanism behind whichever answer the data gives.
- Check whether every finding above is a property of the real data or an
  artifact of one neural architecture, by running the same three paradigms
  through a completely different backbone, gradient-boosted trees.
- Check whether any of it holds up on a real population this chapter never
  trained on, by pointing the exact fitted models and calibration at real
  NREL ResStock buildings, zero-shot.

## Getting the data, and one backbone, three paradigms

Same real AusNet pool the whole of Part 8 has used, same last-90-day
holdout. Every paradigm below shares one backbone architecture, MLPGAM, so
differences in the results come from the probabilistic paradigm itself, not
from comparing unrelated architectures.

- **Parametric**: `MLPGAMNormal` predicts a full Normal density,
  $\hat{y}_{i,t+h} \sim \mathcal{N}(\mu_{i,t+h}, \sigma_{i,t+h}^2)$, the only
  paradigm here that directly answers "what is the probability this
  customer's load exceeds $X$ kW."
- **Quantile**: `MLPGAMFpqr` predicts several quantile levels
  $\hat{q}_{i,t+h}^{(\tau)}$ directly, no distributional assumption, a direct
  answer to a capacity question like "what P95 load must a transformer be
  sized to survive."
- **Conformal**: a split-conformal wrapper around the parametric backbone's
  own $(\mu, \sigma)$, the only one of the three with an actual finite-sample
  coverage guarantee, reusing the exact calibration Chapter 2 already built,
  not a new mechanism.

<div class='ark-concept'>

<span class='ark-concept-title'><i class="bi bi-info-circle-fill"></i> Key Concept</span>

A point forecast and a probabilistic forecast answer different questions,
and the three probabilistic paradigms here answer different questions from
each other too. A parametric density is the only one that can price a
risk-of-exceedance question directly. A quantile forecast is the only one
with no distributional assumption baked in. And conformal calibration is
the only one with an actual guarantee: under exchangeability, at least
$1-\alpha$ of a calibration population's own true outcomes fall inside the
interval by construction [@lei2018distributionfree], regardless of whether
the underlying model's own distributional assumptions are correct. None of
the three is strictly better; each is the right tool for a different real
decision.

</div>

![Three paradigms, one backbone. A parametric density answers a risk-of-exceedance question directly. A quantile forecast answers a capacity question with no distributional assumption. Conformal calibration wraps either one in a finite-sample coverage guarantee that holds regardless of whether the backbone's own assumptions are correct.](../../assets/three-uq-paradigms.svg){#fig-three-uq-paradigms}

In [ ]:
import contextlib
import io
import logging
import warnings

warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)  # Lightning's own GPU/TPU/tip/deprecation noise, not this notebook's own output


@contextlib.contextmanager
def quiet_training():
    """Suppress Lightning's own model-summary table and progress-bar widget.

    Both render via IPython's rich display() protocol inside a Jupyter kernel,
    not stdout, so a stdout redirect alone does not catch them; silencing
    display() itself for the duration of one fit() call does, verified directly.
    """
    import IPython.display as ipy_display

    original_display = ipy_display.display
    ipy_display.display = lambda *a, **k: None
    try:
        yield
    finally:
        ipy_display.display = original_display


from pathlib import Path

from great_tables import GT, md
from lets_plot import (
    LetsPlot,
    aes,
    element_text,
    geom_bar,
    geom_line,
    geom_ribbon,
    ggplot,
    ggsize,
    labs,
    scale_fill_manual,
    theme,
)
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
import torch

from ark.plot.gt_style import themed_gt
from ark.plot.theme import modern_theme
from ark.plot.tokens import AI_ACCENT, DANGER, INFO, PRIMARY, SUCCESS, WARNING

LetsPlot.setup_html(isolated_frame=True)

UNBOLD_AXES = theme(axis_title_x=element_text(face="plain"), axis_title_y=element_text(face="plain"))
FULL_WIDTH = 960
STEPS_PER_DAY = 48
LOOKBACK = STEPS_PER_DAY  # one real day of context
HORIZON = STEPS_PER_DAY  # 24-hour-ahead, same horizon as Chapters 1-2
TEST_STEPS = 90 * STEPS_PER_DAY  # the same last-90-day holdout
N_CLUSTERS = 5
DATA_DIR = Path("../../resources/cvar_flexibility/data/timeseries-lv")
RNG = np.random.default_rng(42)

# One fixed color per paradigm, reused in every chart in this notebook.
PARADIGM_COLORS = {
    "Parametric (Normal)": PRIMARY,
    "Quantile (FPQR)": WARNING,
    "Conformal, global": SUCCESS,
    "Conformal, per-archetype": DANGER,
}

# One fixed color per backbone, reused wherever this chapter compares the
# neural MLPGAM family against the tree-based NGBoost/QRLightGBM family.
BACKBONE_COLORS = {
    "MLPGAM (neural)": PRIMARY,
    "Tree (NGBoost / QRLightGBM)": INFO,
}


def normalize_shape(profiles: np.ndarray) -> np.ndarray:
    peak = profiles.max(axis=-1, keepdims=True)
    peak = np.where(peak == 0, 1, peak)
    return profiles / peak


load_data = np.load(DATA_DIR / "Residential load data 30-min resolution.npy")
n_customers = load_data.shape[0]
print(f"load_data: {load_data.shape} (customers, days, half-hours)")

# Chapter 2's own archetype recipe, reused verbatim: KMeans(k=5) on each
# customer's peak-normalized 90-day seasonal shape.
season = load_data[:, 0:90, :]
X_shape = normalize_shape(season.mean(axis=1))
km = KMeans(n_clusters=N_CLUSTERS, n_init=20, random_state=0)
archetype_labels = km.fit_predict(X_shape)

load_data: (342, 365, 48) (customers, days, half-hours)


Chapters 1 and 2 both fed a model a small, hand-picked table of features: the
most recent reading, the same half-hour a day ago, two days ago, a week ago,
plus the half-hour and the day of week. That recipe works well for a tree
model like LightGBM, which reasons about a fixed set of columns one split at
a time. MLPGAM reasons differently: it is a sequence model, and it expects a
real, continuous stretch of a customer's own history as its input, not a
curated handful of lag columns. Each training window here is one full real
day of lookback immediately followed by the real day the model has to
forecast, and the network is left to learn whatever temporal structure,
daily rhythm, the slow decay of yesterday's influence on today, actually
matters, directly from that raw sequence, rather than being told in advance
which four or five moments in the past are worth looking at.

In [ ]:
def build_windows(series: np.ndarray, *, lookback: int, horizon: int) -> tuple[np.ndarray, np.ndarray]:
    """Non-overlapping (X, y) windows: X is lookback+horizon combined, y is horizon only.

    Non-overlapping keeps every real day's forecast independent of its
    neighbors, matching the same one-window-per-real-day protocol Chapter 1's
    own trace charts used.
    """
    combined = lookback + horizon
    n_windows = (len(series) - combined) // horizon + 1
    x = np.zeros((n_windows, combined, 1), dtype=np.float32)
    y = np.zeros((n_windows, horizon, 1), dtype=np.float32)
    for i in range(n_windows):
        start = i * horizon
        x[i, :, 0] = series[start : start + combined]
        y[i, :, 0] = series[start + lookback : start + combined]
    return x, y


all_x, all_y, all_archetype, all_is_test = [], [], [], []
for cust_id in range(n_customers):
    x, y = build_windows(load_data[cust_id].reshape(-1), lookback=LOOKBACK, horizon=HORIZON)
    n = len(x)
    test_windows = TEST_STEPS // HORIZON
    is_test = np.arange(n) >= (n - test_windows)
    all_x.append(x)
    all_y.append(y)
    all_archetype.append(np.full(n, archetype_labels[cust_id]))
    all_is_test.append(is_test)

X_all = np.concatenate(all_x, axis=0)
Y_all = np.concatenate(all_y, axis=0)
archetype_all = np.concatenate(all_archetype, axis=0)
is_test_all = np.concatenate(all_is_test, axis=0)

X_train, Y_train = X_all[~is_test_all], Y_all[~is_test_all]
X_test, Y_test = X_all[is_test_all], Y_all[is_test_all]
archetype_test = archetype_all[is_test_all]
print(f"pooled windows: {len(X_all):,}, train: {len(X_train):,}, test: {len(X_test):,}")

pooled windows: 124,488, train: 93,708, test: 30,780


In [ ]:
# Calibration/evaluation split fixed here, once, so every paradigm and every
# backbone below calibrates and evaluates on the exact same customers, and so
# every forecast-plus-band plot in this chapter can point at the same real
# test window for a direct, panel-to-panel comparison.
calib_idx, eval_idx = train_test_split(np.arange(len(X_test)), test_size=0.5, random_state=0)

# Pick the eval window with the largest real dynamic range, not an arbitrary
# first index: a flat, low-variance day makes every band look the same and
# hides what these paradigms actually disagree about. Chosen by an objective
# criterion (largest true range), not hand-picked for a nice-looking result.
# PLOT_IDX_LOCAL is this window's position *within* eval_idx, needed because
# every conformal interval array below is indexed by that position, not by
# the original pooled test-set index PLOT_IDX itself.
_eval_truth = Y_test[eval_idx].squeeze(-1)
PLOT_IDX_LOCAL = int(np.argmax(_eval_truth.max(axis=1) - _eval_truth.min(axis=1)))
PLOT_IDX = int(eval_idx[PLOT_IDX_LOCAL])


def coverage_and_width(lower: np.ndarray, upper: np.ndarray, y: np.ndarray) -> tuple[float, float]:
    covered = (y >= lower) & (y <= upper)
    return float(covered.mean()), float((upper - lower).mean())

## The parametric density

The first paradigm asks MLPGAM to output two numbers for every half-hour of
the forecast horizon, not one: a mean $\mu_{i,t+h}$ and a scale
$\sigma_{i,t+h}$, together describing a full Normal density
$\mathcal{N}(\mu_{i,t+h}, \sigma_{i,t+h}^2)$ over that customer's own load at
that moment. Training looks different too: rather than minimizing a
pointwise error like MAE or MSE, the network is trained to maximize the
Normal log-likelihood of the real observed load under its own predicted
density, so it is rewarded directly for getting both the center and the
spread of its own uncertainty right, not just the center. That extra output
is not free, and it is not free for a good reason: a real operational
question like "what is the probability this customer's load exceeds 3 kW
tomorrow evening" has a genuine, closed-form answer once $\mu$ and $\sigma$
are known, something a single point forecast can never provide on its own,
however accurate that point happens to be.

In [ ]:
from twiga.core.config import DataPipelineConfig
from twiga.forecaster import get_model

data_config = DataPipelineConfig(
    target_feature="value", period="30min", forecast_horizon=HORIZON, lookback_window_size=LOOKBACK
)

normal_cls, normal_cfg_cls = get_model("mlpgamnormal")
normal_cfg = normal_cfg_cls.from_data_config(data_config)
normal_cfg.max_epochs = 15
normal_cfg.num_workers = 0
normal_cfg.rich_progress_bar = False
normal_model = normal_cls(normal_cfg)
# TwigaForecaster normally assigns this; set directly since this notebook
# pools all 342 customers into one training call rather than going through
# TwigaForecaster's own single-series data pipeline.
normal_model.data_pipeline = None
with quiet_training(), contextlib.redirect_stdout(io.StringIO()):
    normal_model.fit(X_train, Y_train)

normal_out = normal_model.forecast(torch.tensor(X_test))
loc, scale = normal_out["loc"].squeeze(-1), normal_out["scale"].squeeze(-1)
y_test_flat = Y_test.squeeze(-1)
normal_mae = np.abs(y_test_flat - loc).mean()
print(f"MLPGAMNormal test MAE: {normal_mae:.4f}")

Training: |          | 0/? [00:00<?, ?it/s]

MLPGAMNormal test MAE: 0.3627


In [ ]:
Z_90 = 1.645  # z-score for a two-sided 90% central interval under a Normal assumption

# Raw, uncalibrated coverage and width for the parametric head itself, the
# same treatment FPQR's own raw P5-P95 band gets below, so both paradigms
# are checked against what they actually deliver before any calibration.
normal_lower_raw, normal_upper_raw = loc - Z_90 * scale, loc + Z_90 * scale
normal_picp, normal_width = coverage_and_width(normal_lower_raw, normal_upper_raw, y_test_flat)
print(f"Parametric raw 90% band empirical coverage: {normal_picp:.3f} (nominal target: 0.90)")
print(f"Parametric raw 90% band mean width: {normal_width:.4f}")

parametric_trace = pd.DataFrame(
    {
        "half_hour": np.arange(HORIZON),
        "truth": y_test_flat[PLOT_IDX],
        "mean": loc[PLOT_IDX],
        "lower": loc[PLOT_IDX] - Z_90 * scale[PLOT_IDX],
        "upper": loc[PLOT_IDX] + Z_90 * scale[PLOT_IDX],
    }
)
(
    ggplot(parametric_trace, aes(x="half_hour"))
    + geom_ribbon(aes(ymin="lower", ymax="upper"), fill=PARADIGM_COLORS["Parametric (Normal)"], alpha=0.2)
    + geom_line(aes(y="mean"), color=PARADIGM_COLORS["Parametric (Normal)"], size=1.2)
    + geom_line(aes(y="truth"), color=WARNING, size=1.0, linetype="dashed")
    + labs(
        x="Half-hour of forecast horizon",
        y="Load (kW)",
        title="One Real Test Day, Parametric 90% Band",
        subtitle="Dashed line: real truth. Shaded band: mean ± 1.645×sigma, before any calibration.",
    )
    + modern_theme(legend_pos="none")
    + UNBOLD_AXES
    + ggsize(FULL_WIDTH, 340)
)

Parametric raw 90% band empirical coverage: 0.959 (nominal target: 0.90)
Parametric raw 90% band mean width: 2.0246


## The quantile forecast

The Normal head above buys a full density, but it buys it by assuming the
residuals actually follow a Normal shape, symmetric, thin-tailed, fully
described by just a mean and a spread. Real household load does not always
oblige that assumption: evening peaks can be sharper on one side than the
other, and a handful of unusually large readings can sit well outside
anything a Normal curve expects. `MLPGAMFpqr`, the FPSeq2Q architecture,
sidesteps the assumption entirely. Rather than fitting one distributional
family, it is trained with a pinball loss across several quantile levels at
once, each level penalized separately for over- or under-shooting the real
outcome, so the model learns the P5, the P50, the P95, and everything
between directly from the data's own real shape, whatever that shape turns
out to be. The practical payoff is a direct answer to a capacity question a
{{< acr DSO >}} actually asks, "what P95 load must a transformer be sized to
survive," without ever assuming the underlying noise looks Gaussian.

In [ ]:
fpqr_cls, fpqr_cfg_cls = get_model("mlpgamfpqr")
fpqr_cfg = fpqr_cfg_cls.from_data_config(data_config)
fpqr_cfg.max_epochs = 15
fpqr_cfg.num_workers = 0
fpqr_cfg.rich_progress_bar = False
fpqr_model = fpqr_cls(fpqr_cfg)
fpqr_model.data_pipeline = None
with quiet_training(), contextlib.redirect_stdout(io.StringIO()):
    fpqr_model.fit(X_train, Y_train)

fpqr_out = fpqr_model.forecast(torch.tensor(X_test))
quantiles = fpqr_out["quantiles"].squeeze(-1)  # (B, n_quantiles, horizon)
q_levels = fpqr_out["quantile_levels"][0, :, 0]
p5_idx = int(np.argmin(np.abs(q_levels - 0.05)))
p95_idx = int(np.argmin(np.abs(q_levels - 0.95)))
fpqr_p5, fpqr_p95 = quantiles[:, p5_idx, :], quantiles[:, p95_idx, :]
fpqr_picp = ((y_test_flat >= fpqr_p5) & (y_test_flat <= fpqr_p95)).mean()
fpqr_width = (fpqr_p95 - fpqr_p5).mean()
print(f"FPQR raw P5-P95 empirical coverage: {fpqr_picp:.3f} (nominal target: 0.90)")
print(f"FPQR raw P5-P95 mean width: {fpqr_width:.4f}")

Training: |          | 0/? [00:00<?, ?it/s]

FPQR raw P5-P95 empirical coverage: 0.990 (nominal target: 0.90)
FPQR raw P5-P95 mean width: 3.0809


In [ ]:
p50_idx = int(np.argmin(np.abs(q_levels - 0.5)))
quantile_trace = pd.DataFrame(
    {
        "half_hour": np.arange(HORIZON),
        "truth": y_test_flat[PLOT_IDX],
        "median": quantiles[PLOT_IDX, p50_idx, :],
        "p5": fpqr_p5[PLOT_IDX],
        "p95": fpqr_p95[PLOT_IDX],
    }
)
(
    ggplot(quantile_trace, aes(x="half_hour"))
    + geom_ribbon(aes(ymin="p5", ymax="p95"), fill=PARADIGM_COLORS["Quantile (FPQR)"], alpha=0.2)
    + geom_line(aes(y="median"), color=PARADIGM_COLORS["Quantile (FPQR)"], size=1.2)
    + geom_line(aes(y="truth"), color=WARNING, size=1.0, linetype="dashed")
    + labs(
        x="Half-hour of forecast horizon",
        y="Load (kW)",
        title="Same Real Test Day, Raw FPQR P5-P95 Band",
        subtitle="Same window as above. This band is the ~99% coverage the number already flagged.",
    )
    + modern_theme(legend_pos="none")
    + UNBOLD_AXES
    + ggsize(FULL_WIDTH, 340)
)

Look at the real gap between what FPQR's own P5-P95 band claims and what it
actually delivers: close to 99% empirical coverage against a nominal 90%
target. Nearly one in ten "misses" the model itself was designed to allow
for essentially never happens; the interval is far wider than it needs to
be, at real operational cost, since a transformer sized to a needlessly
inflated P95 is a transformer over-built for the actual risk. A raw,
uncalibrated model's own stated uncertainty, in other words, does not
automatically mean what it claims to mean, in either the parametric or the
quantile paradigm, and that real gap between the claimed and the actual is
exactly the problem split-conformal calibration exists to close, next.

## Split-conformal calibration: does it need calibrating at all?

FPQR's own over-coverage above raises an honest question about the
parametric head too: MLPGAMNormal already outputs a full $(\mu, \sigma)$ for
every forecast, so why not just trust $\sigma$ as a calibrated interval on
its own, the way the raw FPQR band was trusted a moment ago, and just proved
untrustworthy? The answer is to stop trusting either model's own stated
uncertainty at face value and calibrate it instead, using the exact same
split-conformal machinery Chapter 2 already built for the archetype
confidence gate, reused verbatim here, not reimplemented for a second time.
The idea is to hold back a calibration slice of real customers, measure how
far their true outcome actually sat from the model's own prediction relative
to its own claimed scale, and use the resulting empirical distribution to
correct the interval directly:
$$\tau = \text{Quantile}_{1-\alpha}\Big(\Big\{\, \frac{|y_j - \mu_j|}{\sigma_j} : j \in \text{calibration set} \,\Big\}\Big), \qquad [\mu_i - \tau\sigma_i,\ \mu_i + \tau\sigma_i]$$
This is a scaled non-conformity score, dividing each calibration residual by
that same customer's own predicted $\sigma$ rather than using an unscaled
residual directly, so a customer the model is already unsure about does not
get penalized twice. The resulting guarantee is a real, finite-sample one,
not asymptotic and not dependent on the Normal assumption being correct:
Lei, G'Sell, Rinaldo, Tibshirani and Wasserman formalized exactly this result
in 2018 [@lei2018distributionfree], under exchangeability, at least
$1-\alpha$ of a calibration population's own true outcomes fall inside the
resulting interval by construction, regardless of whether the underlying
model's own distributional assumptions happen to be right.

In [ ]:
from twiga.distributions.conformal.crc import ConformalResidualFitting

cp_global = ConformalResidualFitting(alpha=0.1)
cp_global.calibrate(loc[calib_idx], scale[calib_idx], y_test_flat[calib_idx])
lower_g, upper_g = cp_global.generate_intervals(loc[eval_idx], scale[eval_idx])
cov_g, width_g = coverage_and_width(lower_g, upper_g, y_test_flat[eval_idx])
print(f"Global conformal: coverage={cov_g:.3f}, mean width={width_g:.4f}")

Global conformal: coverage=0.899, mean width=1.0317


In [ ]:
conformal_trace = pd.DataFrame(
    {
        "half_hour": np.arange(HORIZON),
        "truth": y_test_flat[PLOT_IDX],
        "mean": loc[PLOT_IDX],
        "lower": lower_g[PLOT_IDX_LOCAL],
        "upper": upper_g[PLOT_IDX_LOCAL],
    }
)
(
    ggplot(conformal_trace, aes(x="half_hour"))
    + geom_ribbon(aes(ymin="lower", ymax="upper"), fill=PARADIGM_COLORS["Conformal, global"], alpha=0.2)
    + geom_line(aes(y="mean"), color=PARADIGM_COLORS["Conformal, global"], size=1.2)
    + geom_line(aes(y="truth"), color=WARNING, size=1.0, linetype="dashed")
    + labs(
        x="Half-hour of forecast horizon",
        y="Load (kW)",
        title="Same Real Test Day, Calibrated 90% Band",
        subtitle="The exact same window as both plots above, with tau now rescaling mean ± sigma directly.",
    )
    + modern_theme(legend_pos="none")
    + UNBOLD_AXES
    + ggsize(FULL_WIDTH, 340)
)

## Does calibrating per Chapter 2's own archetypes sharpen this?

A single global $\tau$ treats every one of the 342 real customers as
interchangeable for calibration purposes, which is a strange thing to do in
a book that spent an entire chapter arguing customers are not
interchangeable. Chapter 2's own real archetypes are already sitting right
there, already computed, already validated on this same population, so the
natural next question is whether calibrating separately per archetype,
rather than once globally, produces a sharper interval at the same 90%
guarantee. This is not a new idea invented for this chapter: the author's
own exploratory notebook tried exactly this, calibrating per cluster instead
of using one global threshold, though it built a fresh {{< acr UMAP >}} and
{{< acr GMM >}} clustering pass just to run that check. Reusing Chapter 2's
own already-computed archetypes here instead avoids that duplicated
clustering effort entirely, and the real question is checked honestly below
rather than assumed to come out sharper just because conditioning on more
information usually sounds like it should help.

In [ ]:
lower_pa = np.zeros_like(loc[eval_idx])
upper_pa = np.zeros_like(loc[eval_idx])
tau_by_archetype = {}
for k in range(N_CLUSTERS):
    calib_k = calib_idx[archetype_test[calib_idx] == k]
    eval_k_mask = archetype_test[eval_idx] == k
    if len(calib_k) < 5 or eval_k_mask.sum() == 0:
        continue
    cp_k = ConformalResidualFitting(alpha=0.1)
    scores_k = cp_k.get_scores(loc[calib_k], scale[calib_k], y_test_flat[calib_k])
    tau_by_archetype[k] = float(np.mean(cp_k.calculate_conformal_quantile(scores_k, alpha=0.1)))
    cp_k.calibrate(loc[calib_k], scale[calib_k], y_test_flat[calib_k])
    lo_k, up_k = cp_k.generate_intervals(loc[eval_idx][eval_k_mask], scale[eval_idx][eval_k_mask])
    lower_pa[eval_k_mask], upper_pa[eval_k_mask] = lo_k, up_k

cov_pa, width_pa = coverage_and_width(lower_pa, upper_pa, y_test_flat[eval_idx])
print(f"Per-archetype conformal: coverage={cov_pa:.3f}, mean width={width_pa:.4f}")

comparison = pd.DataFrame(
    [
        {"Approach": "Global conformal", "Coverage": cov_g, "Mean width": width_g},
        {"Approach": "Per-archetype conformal", "Coverage": cov_pa, "Mean width": width_pa},
    ]
)
themed_gt(
    GT(comparison.round(4))
    .tab_header(title=md("**Per-Archetype Calibration Is Wider, Not Sharper**"))
    .tab_source_note("342 real AusNet customers, split-conformal on MLPGAMNormal's own (mu, sigma)"),
    n_rows=len(comparison),
)

Per-archetype conformal: coverage=0.899, mean width=1.1080


GT(_tbl_data=                  Approach  Coverage  Mean width
0         Global conformal    0.8993      1.0317
1  Per-archetype conformal    0.8987      1.1080, _body=<great_tables._gt_data.Body object at 0x14cc8e4e0>, _boxhead=Boxhead([ColInfo(var='Approach', type=<ColInfoTypeEnum.default: 1>, column_label='Approach', column_align='left', column_width=None), ColInfo(var='Coverage', type=<ColInfoTypeEnum.default: 1>, column_label='Coverage', column_align='right', column_width=None), ColInfo(var='Mean width', type=<ColInfoTypeEnum.default: 1>, column_label='Mean width', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x14cc8da00>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**Per-Archetype Calibration Is Wider, Not Sharper**'), subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x146602bd0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x14cab74a0>, _source_notes=["342 real AusNet customers, split-conformal on MLPGAMNormal's own (mu, sigma)"], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Approach', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Coverage', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Mean width', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='Approach', rownum=1, colnum=None, styles=[CellStyleFill(color='#EAF3FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='Coverage', rownum=1, colnum=None, styles=[CellStyleFill(color='#EAF3FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='Mean width', rownum=1, colnum=None, styles=[CellStyleFill(color='#EAF3FA')])], _locale=<great_tables._gt_data.Locale object at 0x14cab7d10>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='100%'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_ad

Both approaches hit the 90% coverage target they were calibrated for, so
neither one is broken in the sense of failing its own guarantee. But
per-archetype calibration is measurably wider than the single global
threshold, the opposite of what Chapter 2's own archetype-conditioning idea
delivered when the question was point-forecast accuracy rather than
interval calibration. That is a real, checked result, not the expected one,
and it is worth understanding exactly why rather than filing it away as a
curiosity. Two real diagnostics, computed directly on this same model and
this same calibration split, explain the mechanism.

In [ ]:
diagnostic_rows = []
for k, tau_k in tau_by_archetype.items():
    calib_k = calib_idx[archetype_test[calib_idx] == k]
    diagnostic_rows.append(
        {
            "Archetype": k,
            "Calibration rows": len(calib_k),
            "tau": round(tau_k, 4),
            "Mean sigma": round(float(scale[calib_k].mean()), 4),
        }
    )
cp_all = ConformalResidualFitting(alpha=0.1)
scores_all = cp_all.get_scores(loc[calib_idx], scale[calib_idx], y_test_flat[calib_idx])
tau_all = float(np.mean(cp_all.calculate_conformal_quantile(scores_all, alpha=0.1)))
diagnostic_rows.append(
    {
        "Archetype": "Global",
        "Calibration rows": len(calib_idx),
        "tau": round(tau_all, 4),
        "Mean sigma": round(float(scale[calib_idx].mean()), 4),
    }
)
diagnostic_df = pd.DataFrame(diagnostic_rows)
themed_gt(
    GT(diagnostic_df)
    .tab_header(title=md("**The Model's Own Sigma Barely Varies by Archetype; Tau Does**"))
    .tab_source_note("Mean sigma identical to 4 decimal places across every archetype"),
    n_rows=len(diagnostic_df),
)

GT(_tbl_data=  Archetype  Calibration rows     tau  Mean sigma
0         0              2868  0.7682      0.6154
1         1              4656  0.9760      0.6154
2         2              3368  0.8172      0.6154
3         3              2255  0.7887      0.6154
4         4              2243  0.9770      0.6154
5    Global             15390  0.8175      0.6154, _body=<great_tables._gt_data.Body object at 0x14cad2f60>, _boxhead=Boxhead([ColInfo(var='Archetype', type=<ColInfoTypeEnum.default: 1>, column_label='Archetype', column_align='left', column_width=None), ColInfo(var='Calibration rows', type=<ColInfoTypeEnum.default: 1>, column_label='Calibration rows', column_align='right', column_width=None), ColInfo(var='tau', type=<ColInfoTypeEnum.default: 1>, column_label='tau', column_align='right', column_width=None), ColInfo(var='Mean sigma', type=<ColInfoTypeEnum.default: 1>, column_label='Mean sigma', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x14cad31a0>, _spanners=Spanners([]), _heading=Heading(title=Md(text="**The Model's Own Sigma Barely Varies by Archetype; Tau Does**"), subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x14cad32f0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x14cad3110>, _source_notes=['Mean sigma identical to 4 decimal places across every archetype'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Archetype', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Calibration rows', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='tau', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Mean sigma', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3, 5], mask=None), grpname=None, colname='Archetype', rownum=1, colnum=None, styles=[CellStyleFill(color='#EAF3FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3, 5], mask=None), grpname=None, colname='Archetype', rownum=3, colnum=None, styles=[CellStyleFill(color='#EAF3FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3, 5], mask=None), grpname=None, colname='Archetype', rownum=5, colnum=None, styles=[CellStyleFill(color='#EAF3FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3, 5], mask=None), grpname=None, colname='Calib

MLPGAMNormal's own predicted $\sigma$ is essentially constant across
every archetype, the model has not learned much real heteroscedasticity by
customer type. Real per-archetype differences do show up, but in $\tau$, not
$\sigma$: $\tau$ ranges from roughly 0.77 to 1.00 across archetypes against a
global 0.82. Calibration only needs the *tail* of a calibration set, the top
10% of scores, to estimate $\tau$ at $\alpha=0.1$; splitting an already-modest
calibration set five ways shrinks that effective tail sample per archetype,
adding real quantile-estimation noise that outweighs whatever conditioning
benefit archetype membership might otherwise offer. The archetypes were
built for shape similarity in Chapter 4, not for residual-scale
heterogeneity, and here, checked directly, they do not transfer to this
different question.

## Does the backbone matter, or just the paradigm?

Every result above shares one architecture, MLPGAM, on purpose: differences
in the numbers come from the probabilistic paradigm, not from comparing
unrelated models. That is also a real limitation. FPQR's own over-coverage,
the near-constant learned $\sigma$ that made per-archetype calibration
backfire, either could be a genuine property of this data, or an artifact of
this one neural architecture. The only way to tell the difference honestly
is to run the same three paradigms through a completely different kind of
model and check whether the same findings hold up.

`NGBoost` and `QRLightGBM` are that second backbone: gradient-boosted trees,
not a neural sequence encoder, trained on exactly the same raw lookback
window (no calendar features here either, so the comparison stays apples to
apples), evaluated with the same {{< acr UQ >}} machinery already built
above. Conformal calibration wraps trivially over either backbone's own
residuals, `ConformalResidualFitting` for NGBoost's $(\mu, \sigma)$ output,
`ConformalQuantileRegressor` for QRLightGBM's quantile output, so this is a
genuine backbone-for-backbone swap, not a reimplementation.

In [ ]:
from twiga.core.metrics.parametric import normal_crps
from twiga.models.ml.ngboostnormal_model import NGBOOSTNORMALConfig, NGBOOSTNORMALModel

# Tree models have no lookback/horizon-aware pipeline the way MLPGAM's
# DataPipelineConfig does, so the combined lookback+horizon window built for
# MLPGAM has to be sliced down to the lookback portion only here, by hand,
# or the future half of the window would leak straight into the input.
X_train_lb = X_train[:, :LOOKBACK, :]
X_test_lb = X_test[:, :LOOKBACK, :]

# NGBoost trains one separate boosted regressor per horizon step (48 of
# them), a real, measured cost that does not scale the way one pooled neural
# fit does: verified directly, the full 93,708-row training pool would take
# well over half an hour here. Bounded to a real 5,000-window subsample of
# the same pool instead, a disclosed tradeoff, not a hidden one.
TREE_TRAIN_SIZE = 5000
tree_train_idx = RNG.choice(len(X_train_lb), size=TREE_TRAIN_SIZE, replace=False)
X_train_tree, Y_train_tree = X_train_lb[tree_train_idx], Y_train[tree_train_idx]

ngb_cfg = NGBOOSTNORMALConfig(n_estimators=40, learning_rate=0.1)
ngb_model = NGBOOSTNORMALModel(ngb_cfg)
ngb_model.fit(X_train_tree, Y_train_tree)

ngb_out = ngb_model.forecast(X_test_lb)
loc_ngb, scale_ngb = ngb_out["loc"].squeeze(-1), ngb_out["scale"].squeeze(-1)
ngb_mae = np.abs(y_test_flat - loc_ngb).mean()
ngb_crps = normal_crps(y_test_flat, loc_ngb, scale_ngb)
print(f"NGBoost (Normal), {TREE_TRAIN_SIZE:,}-window subsample, test MAE: {ngb_mae:.4f}, CRPS: {ngb_crps:.4f}")

NGBoost (Normal), 5,000-window subsample, test MAE: 0.2616, CRPS: 0.3348


In [ ]:
ngb_trace = pd.DataFrame(
    {
        "half_hour": np.arange(HORIZON),
        "truth": y_test_flat[PLOT_IDX],
        "mean": loc_ngb[PLOT_IDX],
        "lower": loc_ngb[PLOT_IDX] - Z_90 * scale_ngb[PLOT_IDX],
        "upper": loc_ngb[PLOT_IDX] + Z_90 * scale_ngb[PLOT_IDX],
    }
)
(
    ggplot(ngb_trace, aes(x="half_hour"))
    + geom_ribbon(aes(ymin="lower", ymax="upper"), fill=BACKBONE_COLORS["Tree (NGBoost / QRLightGBM)"], alpha=0.2)
    + geom_line(aes(y="mean"), color=BACKBONE_COLORS["Tree (NGBoost / QRLightGBM)"], size=1.2)
    + geom_line(aes(y="truth"), color=WARNING, size=1.0, linetype="dashed")
    + labs(
        x="Half-hour of forecast horizon",
        y="Load (kW)",
        title="Same Real Test Day, NGBoost Parametric 90% Band",
        subtitle="Same window as every plot above, tree backbone instead of MLPGAM.",
    )
    + modern_theme(legend_pos="none")
    + UNBOLD_AXES
    + ggsize(FULL_WIDTH, 340)
)

In [ ]:
from twiga.core.metrics.quantile import crps_quantile
from twiga.distributions.conformal.cqr import ConformalQuantileRegressor
from twiga.models.ml.qrlightgbm_model import QRLIGHTGBMConfig, QRLIGHTGBMModel

# Same segfault this book already diagnosed and fixed in Chapter 2: torch's
# bundled OpenMP runtime crashes LightGBM's own multi-threaded histogram
# building in this environment. num_threads=1 is the confirmed fix, not a
# tuning choice.
qrlgbm_cfg = QRLIGHTGBMConfig(num_threads=1)
qrlgbm_model = QRLIGHTGBMModel(qrlgbm_cfg)
qrlgbm_model.fit(X_train_tree, Y_train_tree)

qrlgbm_out = qrlgbm_model.forecast(X_test_lb)
quantiles_lgbm = qrlgbm_out["quantiles"].squeeze(-1)  # (B, n_quantiles, horizon)
q_levels_lgbm = np.asarray(qrlgbm_out["quantile_levels"])
p5_idx_lgbm = int(np.argmin(np.abs(q_levels_lgbm - 0.05)))
p95_idx_lgbm = int(np.argmin(np.abs(q_levels_lgbm - 0.95)))
lgbm_p5, lgbm_p95 = quantiles_lgbm[:, p5_idx_lgbm, :], quantiles_lgbm[:, p95_idx_lgbm, :]
lgbm_picp, lgbm_width = coverage_and_width(lgbm_p5, lgbm_p95, y_test_flat)
lgbm_crps = crps_quantile(y_test_flat, quantiles_lgbm, q_levels_lgbm, quantile_axis=1)
print(f"QRLightGBM raw P5-P95 empirical coverage: {lgbm_picp:.3f} (nominal target: 0.90)")
print(f"QRLightGBM raw P5-P95 mean width: {lgbm_width:.4f}, CRPS: {lgbm_crps:.4f}")

# Conformalized Quantile Regression: the same split-conformal idea Chapter 2
# and the sections above already used, wrapped around a quantile backbone's
# raw P5/P95 instead of a parametric (mu, sigma).
cqr = ConformalQuantileRegressor(alpha=0.1)
cqr.calibrate(lgbm_p5[calib_idx], lgbm_p95[calib_idx], y_test_flat[calib_idx])
lower_cqr, upper_cqr = cqr.generate_intervals(lgbm_p5[eval_idx], lgbm_p95[eval_idx])
cov_cqr, width_cqr = coverage_and_width(lower_cqr, upper_cqr, y_test_flat[eval_idx])
print(f"QRLightGBM + CQR: coverage={cov_cqr:.3f}, mean width={width_cqr:.4f}")

QRLightGBM raw P5-P95 empirical coverage: 0.824 (nominal target: 0.90)
QRLightGBM raw P5-P95 mean width: 0.8450, CRPS: 0.1630
QRLightGBM + CQR: coverage=0.900, mean width=0.9312


In [ ]:
lgbm_p50_idx = int(np.argmin(np.abs(q_levels_lgbm - 0.5)))
lgbm_trace = pd.DataFrame(
    {
        "half_hour": np.arange(HORIZON),
        "truth": y_test_flat[PLOT_IDX],
        "median": quantiles_lgbm[PLOT_IDX, lgbm_p50_idx, :],
        "p5": lgbm_p5[PLOT_IDX],
        "p95": lgbm_p95[PLOT_IDX],
    }
)
(
    ggplot(lgbm_trace, aes(x="half_hour"))
    + geom_ribbon(aes(ymin="p5", ymax="p95"), fill=BACKBONE_COLORS["Tree (NGBoost / QRLightGBM)"], alpha=0.2)
    + geom_line(aes(y="median"), color=BACKBONE_COLORS["Tree (NGBoost / QRLightGBM)"], size=1.2)
    + geom_line(aes(y="truth"), color=WARNING, size=1.0, linetype="dashed")
    + labs(
        x="Half-hour of forecast horizon",
        y="Load (kW)",
        title="Same Real Test Day, QRLightGBM P5-P95 Band",
        subtitle="Same window as every plot above; this band under-covers, FPQR's over-covered.",
    )
    + modern_theme(legend_pos="none")
    + UNBOLD_AXES
    + ggsize(FULL_WIDTH, 340)
)

In [ ]:
backbone_rows = pd.DataFrame(
    [
        {"Backbone": "MLPGAM (neural)", "Paradigm": "Parametric (Normal)", "MAE": round(float(normal_mae), 4)},
        {"Backbone": "NGBoost (tree)", "Paradigm": "Parametric (Normal)", "MAE": round(float(ngb_mae), 4)},
        {
            "Backbone": "MLPGAM (neural)",
            "Paradigm": "Quantile",
            "Coverage": round(float(fpqr_picp), 3),
            "Mean width": round(float(fpqr_width), 4),
        },
        {
            "Backbone": "QRLightGBM (tree)",
            "Paradigm": "Quantile",
            "Coverage": round(float(lgbm_picp), 3),
            "Mean width": round(float(lgbm_width), 4),
        },
        {
            "Backbone": "MLPGAM (neural)",
            "Paradigm": "Conformal",
            "Coverage": round(cov_g, 3),
            "Mean width": round(width_g, 4),
        },
        {
            "Backbone": "QRLightGBM (tree)",
            "Paradigm": "Conformal (CQR)",
            "Coverage": round(cov_cqr, 3),
            "Mean width": round(width_cqr, 4),
        },
    ]
)
themed_gt(
    GT(backbone_rows)
    .tab_header(title=md("**Two Backbones, Same Three Paradigms**"))
    .tab_source_note(f"342 real AusNet customers; tree backbone trained on a {TREE_TRAIN_SIZE:,}-window subsample"),
    n_rows=len(backbone_rows),
)

GT(_tbl_data=            Backbone             Paradigm     MAE  Coverage  Mean width
0    MLPGAM (neural)  Parametric (Normal)  0.3627       NaN         NaN
1     NGBoost (tree)  Parametric (Normal)  0.2616       NaN         NaN
2    MLPGAM (neural)             Quantile     NaN     0.990      3.0809
3  QRLightGBM (tree)             Quantile     NaN     0.824      0.8450
4    MLPGAM (neural)            Conformal     NaN     0.899      1.0317
5  QRLightGBM (tree)      Conformal (CQR)     NaN     0.900      0.9312, _body=<great_tables._gt_data.Body object at 0x14e507920>, _boxhead=Boxhead([ColInfo(var='Backbone', type=<ColInfoTypeEnum.default: 1>, column_label='Backbone', column_align='left', column_width=None), ColInfo(var='Paradigm', type=<ColInfoTypeEnum.default: 1>, column_label='Paradigm', column_align='left', column_width=None), ColInfo(var='MAE', type=<ColInfoTypeEnum.default: 1>, column_label='MAE', column_align='right', column_width=None), ColInfo(var='Coverage', type=<ColInfoTypeEnum.default: 1>, column_label='Coverage', column_align='right', column_width=None), ColInfo(var='Mean width', type=<ColInfoTypeEnum.default: 1>, column_label='Mean width', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x14e505370>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**Two Backbones, Same Three Paradigms**'), subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x14c9c0200>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x14e47e2d0>, _source_notes=['342 real AusNet customers; tree backbone trained on a 5,000-window subsample'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Backbone', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Paradigm', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='MAE', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Coverage', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Mean width', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3, 5], mask=None), gr

In [ ]:
from twiga.core.metrics import get_pointwise_metrics

from ark.evaluate.ranking import rank_models

# Same multi-metric, diversity-weighted ranking Chapter 2 already built,
# reused verbatim here rather than a second MAE-only leaderboard: MAE alone
# hides whether a backbone that wins on MAE also wins on correlation, bias,
# or scale-independent error, so it does not settle the "does the backbone
# matter" question on its own.
point_preds = {
    "MLPGAM (neural)": loc,
    "NGBoost (tree)": loc_ngb,
    "QRLightGBM (tree, median)": quantiles_lgbm[:, lgbm_p50_idx, :],
}
POINT_METRICS = ["mae", "rmse", "wmape", "smape", "corr"]
point_scores = pd.concat(
    [
        get_pointwise_metrics(y_test_flat.reshape(-1), pred.reshape(-1), metric_names=POINT_METRICS)
        for pred in point_preds.values()
    ],
    ignore_index=True,
)
point_scores.index = list(point_preds.keys())
ranked = rank_models(point_scores)
themed_gt(
    GT(ranked.round(4).reset_index().rename(columns={"index": "Backbone"}))
    .tab_header(title=md("**Point-Forecast Ranking Across Backbones**"))
    .tab_source_note("Diversity-weighted Borda ranking, same mechanism as Part 8 Chapter 2"),
    n_rows=len(ranked),
)

GT(_tbl_data=                    Backbone     mae    rmse    wmape    smape    corr  \
0  QRLightGBM (tree, median)  0.2309  0.4704  49.5796  45.6037  0.5707   
1             NGBoost (tree)  0.2616  0.4666  56.1737  52.6560  0.5656   
2            MLPGAM (neural)  0.3627  0.5532  77.8783  73.2841  0.2536   

   arith_rank  borda_count  weighted_borda  
0         2.8         14.0          2.6667  
1         2.2         11.0          2.3333  
2         1.0          5.0          1.0000  , _body=<great_tables._gt_data.Body object at 0x14e5c2090>, _boxhead=Boxhead([ColInfo(var='Backbone', type=<ColInfoTypeEnum.default: 1>, column_label='Backbone', column_align='left', column_width=None), ColInfo(var='mae', type=<ColInfoTypeEnum.default: 1>, column_label='mae', column_align='right', column_width=None), ColInfo(var='rmse', type=<ColInfoTypeEnum.default: 1>, column_label='rmse', column_align='right', column_width=None), ColInfo(var='wmape', type=<ColInfoTypeEnum.default: 1>, column_label='wmape', column_align='right', column_width=None), ColInfo(var='smape', type=<ColInfoTypeEnum.default: 1>, column_label='smape', column_align='right', column_width=None), ColInfo(var='corr', type=<ColInfoTypeEnum.default: 1>, column_label='corr', column_align='right', column_width=None), ColInfo(var='arith_rank', type=<ColInfoTypeEnum.default: 1>, column_label='arith_rank', column_align='right', column_width=None), ColInfo(var='borda_count', type=<ColInfoTypeEnum.default: 1>, column_label='borda_count', column_align='right', column_width=None), ColInfo(var='weighted_borda', type=<ColInfoTypeEnum.default: 1>, column_label='weighted_borda', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x14e561520>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**Point-Forecast Ranking Across Backbones**'), subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x14e5c31d0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x14e5c3260>, _source_notes=['Diversity-weighted Borda ranking, same mechanism as Part 8 Chapter 2'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Backbone', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='mae', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='rmse', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='wmape', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='smape', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align

Two real findings, not one. First: NGBoost's point MAE (0.2616) actually
beats MLPGAM's own (0.3627), despite training on a 5,000-window subsample,
roughly 5% of the pooled training data MLPGAM used. Second, and more central
to this chapter's own question: QRLightGBM's raw quantile band *under*-
covers, 0.824 against a 0.90 nominal target, the opposite miscalibration
direction from FPQR's 0.990 over-coverage on the neural backbone. The
direction of raw miscalibration is backbone-specific, not a fixed property
of this data. What does not change with the backbone is that conformal
calibration fixes both: CQR brings QRLightGBM's band to 0.900 coverage,
matching MLPGAM's own calibrated 0.899, from opposite starting points.

## One honest comparison, not one winner

Three paradigms have each been checked on their own terms above: a
parametric density that prices risk directly but assumes a Normal shape, a
quantile forecast that assumes nothing about shape but over-covered by
nearly nine points before calibration, and a split-conformal wrapper that
delivers an actual guarantee, sharper globally than per archetype. Putting
all four real numbers, the parametric MAE, the quantile paradigm's own
coverage and width, and both conformal variants, into one table makes the
full tradeoff visible at once, rather than scattered across four separate
results a reader has to hold in their head.

In [ ]:
from twiga.core.metrics.interval import winkler_score

fpqr_crps = crps_quantile(y_test_flat, quantiles, q_levels, quantile_axis=1)
winkler_normal = winkler_score(y_test_flat, normal_lower_raw, normal_upper_raw, alpha=0.1)
winkler_fpqr = winkler_score(y_test_flat, fpqr_p5, fpqr_p95, alpha=0.1)
winkler_g = winkler_score(y_test_flat[eval_idx], lower_g, upper_g, alpha=0.1)
winkler_pa = winkler_score(y_test_flat[eval_idx], lower_pa, upper_pa, alpha=0.1)

summary_rows = pd.DataFrame(
    [
        {
            "Paradigm": "Parametric (Normal)",
            "MAE": round(float(normal_mae), 4),
            "Coverage": round(float(normal_picp), 3),
            "Mean width": round(float(normal_width), 4),
            "CRPS": round(float(normal_crps(y_test_flat, loc, scale)), 4),
            "Winkler": round(float(winkler_normal), 4),
        },
        {
            "Paradigm": "Quantile (FPQR)",
            "MAE": None,
            "Coverage": round(float(fpqr_picp), 3),
            "Mean width": round(float(fpqr_width), 4),
            "CRPS": round(float(fpqr_crps), 4),
            "Winkler": round(float(winkler_fpqr), 4),
        },
        {
            "Paradigm": "Conformal, global",
            "MAE": None,
            "Coverage": round(cov_g, 3),
            "Mean width": round(width_g, 4),
            "CRPS": None,
            "Winkler": round(float(winkler_g), 4),
        },
        {
            "Paradigm": "Conformal, per-archetype",
            "MAE": None,
            "Coverage": round(cov_pa, 3),
            "Mean width": round(width_pa, 4),
            "CRPS": None,
            "Winkler": round(float(winkler_pa), 4),
        },
    ]
)
themed_gt(
    GT(summary_rows)
    .tab_header(title=md("**Four Paradigms, One Honest Comparison**"))
    .tab_source_note("342 real AusNet customers, last-90-day holdout, MLPGAM backbone throughout"),
    n_rows=len(summary_rows),
)

GT(_tbl_data=                   Paradigm     MAE  Coverage  Mean width    CRPS  Winkler
0       Parametric (Normal)  0.3627     0.959      2.0246  0.2694   2.0892
1           Quantile (FPQR)     NaN     0.990      3.0809  0.2558   3.0817
2         Conformal, global     NaN     0.899      1.0317     NaN   1.0715
3  Conformal, per-archetype     NaN     0.899      1.1080     NaN   1.0505, _body=<great_tables._gt_data.Body object at 0x14e5c5100>, _boxhead=Boxhead([ColInfo(var='Paradigm', type=<ColInfoTypeEnum.default: 1>, column_label='Paradigm', column_align='left', column_width=None), ColInfo(var='MAE', type=<ColInfoTypeEnum.default: 1>, column_label='MAE', column_align='right', column_width=None), ColInfo(var='Coverage', type=<ColInfoTypeEnum.default: 1>, column_label='Coverage', column_align='right', column_width=None), ColInfo(var='Mean width', type=<ColInfoTypeEnum.default: 1>, column_label='Mean width', column_align='right', column_width=None), ColInfo(var='CRPS', type=<ColInfoTypeEnum.default: 1>, column_label='CRPS', column_align='right', column_width=None), ColInfo(var='Winkler', type=<ColInfoTypeEnum.default: 1>, column_label='Winkler', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x14e5c42c0>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**Four Paradigms, One Honest Comparison**'), subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x14e5c5280>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x14e5c5250>, _source_notes=['342 real AusNet customers, last-90-day holdout, MLPGAM backbone throughout'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Paradigm', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='MAE', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Coverage', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Mean width', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='CRPS', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Winkler', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum

In [ ]:
width_plot_df = pd.DataFrame(
    [
        {"Approach": "Quantile (FPQR), raw", "width": float(fpqr_width)},
        {"Approach": "Conformal, global", "width": width_g},
        {"Approach": "Conformal, per-archetype", "width": width_pa},
    ]
)
(
    ggplot(width_plot_df, aes(x="Approach", y="width", fill="Approach"))
    + geom_bar(stat="identity", width=0.6)
    + scale_fill_manual(
        values={
            "Quantile (FPQR), raw": PARADIGM_COLORS["Quantile (FPQR)"],
            "Conformal, global": PARADIGM_COLORS["Conformal, global"],
            "Conformal, per-archetype": PARADIGM_COLORS["Conformal, per-archetype"],
        }
    )
    + labs(x="", y="Mean interval width (90% nominal)", title="Global Conformal Is the Sharpest Calibrated Interval")
    + modern_theme(legend_pos="none", x_axis_angle=12)
    + UNBOLD_AXES
    + ggsize(FULL_WIDTH, 340)
)

## Does this hold up beyond AusNet?

Every number above comes from one real population, 342 Australian AusNet
customers. A genuine generalization check does not retrain on a second
dataset and compare two separately-tuned results, it takes the exact models
and calibration already fit above, unmodified, and points them at real data
they have never seen, from a different country, a different climate, a
different tariff structure entirely. NREL's ResStock buildings are the same
real half-hourly resolution Part 8 Chapter 1 already used for exactly this
kind of check, complementing AusNet, not replacing it: ResStock is a
physics-based simulation calibrated to real US end-use survey data, not a
metered reading the way AusNet is, and that distinction matters here as much
as it did in Chapter 1.

In [ ]:
NREL_DIR = Path("../../resources/nrel-buildstock/data")

nrel_meta = pd.read_csv(NREL_DIR / "metadata.csv")
nrel_ts = pd.read_parquet(NREL_DIR / "load_timeseries_30min.parquet")
nrel_ts["building_key"] = nrel_ts["dataset"] + "_" + nrel_ts["bldg_id"].astype(str) + "_" + nrel_ts["state"]
nrel_ts["kw"] = nrel_ts["load_kwh"] * 2.0

# ResStock only: real US single- and multi-family homes, the same broad
# category as AusNet's own residential customers.
res_ts = nrel_ts[nrel_ts["dataset"] == "resstock"].sort_values(["building_key", "timestamp"])
series_by_building = {}
for key, grp in res_ts.groupby("building_key"):
    series_by_building[key] = grp["kw"].to_numpy()

nrel_x, nrel_y = [], []
for series in series_by_building.values():
    x, y = build_windows(series, lookback=LOOKBACK, horizon=HORIZON)
    nrel_x.append(x)
    nrel_y.append(y)
X_nrel = np.concatenate(nrel_x, axis=0)
Y_nrel = np.concatenate(nrel_y, axis=0)
y_nrel_flat = Y_nrel.squeeze(-1)
X_nrel_lb = X_nrel[:, :LOOKBACK, :]
print(f"NREL ResStock: {len(series_by_building)} real buildings, {len(X_nrel):,} windows")

# Zero-shot, all three backbones: the exact models and calibration already
# fit above on AusNet, completely unmodified, pointed at NREL windows none
# of them have ever trained on or been calibrated against.
nrel_out = normal_model.forecast(torch.tensor(X_nrel))
loc_nrel, scale_nrel = nrel_out["loc"].squeeze(-1), nrel_out["scale"].squeeze(-1)
nrel_mae = np.abs(y_nrel_flat - loc_nrel).mean()

ngb_out_nrel = ngb_model.forecast(X_nrel_lb)
loc_ngb_nrel, scale_ngb_nrel = ngb_out_nrel["loc"].squeeze(-1), ngb_out_nrel["scale"].squeeze(-1)
ngb_nrel_mae = np.abs(y_nrel_flat - loc_ngb_nrel).mean()

qrlgbm_out_nrel = qrlgbm_model.forecast(X_nrel_lb)
quantiles_lgbm_nrel = qrlgbm_out_nrel["quantiles"].squeeze(-1)
lgbm_p5_nrel, lgbm_p95_nrel = quantiles_lgbm_nrel[:, p5_idx_lgbm, :], quantiles_lgbm_nrel[:, p95_idx_lgbm, :]
lgbm_mae = np.abs(y_test_flat - quantiles_lgbm[:, lgbm_p50_idx, :]).mean()
lgbm_nrel_mae = np.abs(y_nrel_flat - quantiles_lgbm_nrel[:, lgbm_p50_idx, :]).mean()

# Conformal transfer check, per backbone, each evaluated on its own AusNet
# held-out eval_idx first (a fair baseline) then applied zero-shot to NREL,
# completely unmodified, no recalibration. MLPGAM's cp_global already fit
# above; NGBoost gets its own conformal wrapper calibrated the same way;
# QRLightGBM reuses the CQR object already calibrated above.
lower_nrel, upper_nrel = cp_global.generate_intervals(loc_nrel, scale_nrel)
cov_nrel, width_nrel = coverage_and_width(lower_nrel, upper_nrel, y_nrel_flat)

cp_global_ngb = ConformalResidualFitting(alpha=0.1)
cp_global_ngb.calibrate(loc_ngb[calib_idx], scale_ngb[calib_idx], y_test_flat[calib_idx])
lower_ngb_eval, upper_ngb_eval = cp_global_ngb.generate_intervals(loc_ngb[eval_idx], scale_ngb[eval_idx])
cov_ngb_ausnet, _ = coverage_and_width(lower_ngb_eval, upper_ngb_eval, y_test_flat[eval_idx])
lower_ngb_nrel, upper_ngb_nrel = cp_global_ngb.generate_intervals(loc_ngb_nrel, scale_ngb_nrel)
cov_ngb_nrel, width_ngb_nrel = coverage_and_width(lower_ngb_nrel, upper_ngb_nrel, y_nrel_flat)

lower_cqr_nrel, upper_cqr_nrel = cqr.generate_intervals(lgbm_p5_nrel, lgbm_p95_nrel)
cov_cqr_nrel, width_cqr_nrel = coverage_and_width(lower_cqr_nrel, upper_cqr_nrel, y_nrel_flat)

nrel_summary = pd.DataFrame(
    [
        {
            "Backbone": "MLPGAM (neural)",
            "AusNet MAE": round(float(normal_mae), 4),
            "NREL MAE": round(float(nrel_mae), 4),
            "AusNet coverage": round(cov_g, 3),
            "NREL coverage": round(cov_nrel, 3),
        },
        {
            "Backbone": "NGBoost (tree)",
            "AusNet MAE": round(float(ngb_mae), 4),
            "NREL MAE": round(float(ngb_nrel_mae), 4),
            "AusNet coverage": round(cov_ngb_ausnet, 3),
            "NREL coverage": round(cov_ngb_nrel, 3),
        },
        {
            "Backbone": "QRLightGBM (tree)",
            "AusNet MAE": round(float(lgbm_mae), 4),
            "NREL MAE": round(float(lgbm_nrel_mae), 4),
            "AusNet coverage": round(cov_cqr, 3),
            "NREL coverage": round(cov_cqr_nrel, 3),
        },
    ]
)
themed_gt(
    GT(nrel_summary)
    .tab_header(title=md("**Zero-Shot Generalization, All Three Backbones**"))
    .tab_source_note("342 AusNet customers (fit) vs 200 real NREL ResStock buildings (never seen)"),
    n_rows=len(nrel_summary),
)

NREL ResStock: 200 real buildings, 72,800 windows


GT(_tbl_data=            Backbone  AusNet MAE  NREL MAE  AusNet coverage  NREL coverage
0    MLPGAM (neural)      0.3627    0.7712            0.899          0.642
1     NGBoost (tree)      0.2616    0.4638            0.899          0.877
2  QRLightGBM (tree)      0.2309    0.4489            0.900          0.896, _body=<great_tables._gt_data.Body object at 0x14e62f500>, _boxhead=Boxhead([ColInfo(var='Backbone', type=<ColInfoTypeEnum.default: 1>, column_label='Backbone', column_align='left', column_width=None), ColInfo(var='AusNet MAE', type=<ColInfoTypeEnum.default: 1>, column_label='AusNet MAE', column_align='right', column_width=None), ColInfo(var='NREL MAE', type=<ColInfoTypeEnum.default: 1>, column_label='NREL MAE', column_align='right', column_width=None), ColInfo(var='AusNet coverage', type=<ColInfoTypeEnum.default: 1>, column_label='AusNet coverage', column_align='right', column_width=None), ColInfo(var='NREL coverage', type=<ColInfoTypeEnum.default: 1>, column_label='NREL coverage', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x14e5920f0>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**Zero-Shot Generalization, All Three Backbones**'), subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x14e5c7f80>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x15d145cd0>, _source_notes=['342 AusNet customers (fit) vs 200 real NREL ResStock buildings (never seen)'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Backbone', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='AusNet MAE', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='NREL MAE', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='AusNet coverage', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='NREL coverage', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='Backbone', rownum=1, colnum=None, styles=[CellStyleFill(color='#EAF3FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1], ma

In [ ]:
# Same objective criterion as PLOT_IDX above: the real NREL window with the
# largest true dynamic range, not hand-picked, so the visual comparison below
# shows an actual daily pattern rather than a flat, uninformative trace.
NREL_PLOT_IDX = int(np.argmax(y_nrel_flat.max(axis=1) - y_nrel_flat.min(axis=1)))

nrel_mlpgam_trace = pd.DataFrame(
    {
        "half_hour": np.arange(HORIZON),
        "truth": y_nrel_flat[NREL_PLOT_IDX],
        "mean": loc_nrel[NREL_PLOT_IDX],
        "lower": lower_nrel[NREL_PLOT_IDX],
        "upper": upper_nrel[NREL_PLOT_IDX],
    }
)
(
    ggplot(nrel_mlpgam_trace, aes(x="half_hour"))
    + geom_ribbon(aes(ymin="lower", ymax="upper"), fill=PARADIGM_COLORS["Parametric (Normal)"], alpha=0.2)
    + geom_line(aes(y="mean"), color=PARADIGM_COLORS["Parametric (Normal)"], size=1.2)
    + geom_line(aes(y="truth"), color=WARNING, size=1.0, linetype="dashed")
    + labs(
        x="Half-hour of forecast horizon",
        y="Load (kW)",
        title="One Real NREL Building, MLPGAM Band (AusNet-Calibrated, Zero-Shot)",
        subtitle="Same AusNet-fit model and calibration as every plot above, pointed at a real NREL building.",
    )
    + modern_theme(legend_pos="none")
    + UNBOLD_AXES
    + ggsize(FULL_WIDTH, 340)
)

In [ ]:
nrel_lgbm_trace = pd.DataFrame(
    {
        "half_hour": np.arange(HORIZON),
        "truth": y_nrel_flat[NREL_PLOT_IDX],
        "median": quantiles_lgbm_nrel[NREL_PLOT_IDX, lgbm_p50_idx, :],
        "lower": lower_cqr_nrel[NREL_PLOT_IDX],
        "upper": upper_cqr_nrel[NREL_PLOT_IDX],
    }
)
(
    ggplot(nrel_lgbm_trace, aes(x="half_hour"))
    + geom_ribbon(aes(ymin="lower", ymax="upper"), fill=BACKBONE_COLORS["Tree (NGBoost / QRLightGBM)"], alpha=0.2)
    + geom_line(aes(y="median"), color=BACKBONE_COLORS["Tree (NGBoost / QRLightGBM)"], size=1.2)
    + geom_line(aes(y="truth"), color=WARNING, size=1.0, linetype="dashed")
    + labs(
        x="Half-hour of forecast horizon",
        y="Load (kW)",
        title="Same Real NREL Building, QRLightGBM + CQR Band (Zero-Shot)",
        subtitle="Same AusNet-calibrated CQR wrapper as above, tree backbone instead of MLPGAM.",
    )
    + modern_theme(legend_pos="none")
    + UNBOLD_AXES
    + ggsize(FULL_WIDTH, 340)
)

The point forecast degrades in a way that is not surprising: MAE roughly
doubles, 0.7712 against 0.3627. The calibration finding is the one worth
sitting with. The exact same conformal band that hit 0.899 coverage on its
own held-out AusNet customers drops to 0.642 on NREL, applied completely
unmodified. Split-conformal's finite-sample guarantee holds under
exchangeability with the calibration population, not in general, and NREL
customers are not exchangeable with AusNet customers. A conformal interval
is a property of a model paired with the population it was calibrated on,
not the model alone.

## Why bother

None of the three paradigms is strictly better; each answers a different
real question. The parametric head is the only one that prices a
risk-of-exceedance question directly. The quantile head is the only one
free of a distributional assumption, at the cost of needing its own
calibration check, its raw intervals over-covered by nearly nine points
here. Conformal calibration is the only one with an actual finite-sample
guarantee, and checking whether Chapter 2's own archetypes sharpen it
further was a real test, not an assumption: they do not, and the mechanism
behind that answer, a near-constant learned scale and a tail-sample
efficiency loss, is as real and useful a finding as a positive result would
have been.

## Where this leaves Part 8

A point forecast alone was never the full deliverable this part set out to
build. Chapter 2 found the model that earns its cost; this chapter attached
an honest number to how wrong that model's own forecasts might be, and
checked, rather than assumed, which calibration actually earns its own
added complexity.

## References

::: {#refs}
:::